# 02. KPI Deep Dive & Profitability Analysis - MediSure

This notebook performs a deep-dive analysis into portfolio loss ratio (Incurred Claim Ratio),
unprofitable customer segments (ICR > 100%), and persistency retention drivers.


In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.io import get_duckdb_connection

con = get_duckdb_connection(read_only=True)


## 1. Portfolio Loss Ratio by Plan Tier & Type

In [ ]:
prof_df = con.execute("""
    WITH policy_exp AS (
        SELECT policy_id, plan_type, plan_tier,
               annual_premium * ((DATE_DIFF('day', policy_start_date, policy_end_date) + 1.0) / 365.25) AS earned_prem
        FROM dim_policy
    ),
    claims_agg AS (
        SELECT policy_id, SUM(paid_amount) AS incurred
        FROM fact_claims GROUP BY policy_id
    )
    SELECT p.plan_type, p.plan_tier,
           ROUND(SUM(p.earned_prem), 2) AS earned_premium,
           ROUND(COALESCE(SUM(c.incurred), 0.0), 2) AS incurred_claims,
           ROUND(COALESCE(SUM(c.incurred), 0.0) / SUM(p.earned_prem) * 100.0, 2) AS loss_ratio_pct
    FROM policy_exp p
    LEFT JOIN claims_agg c ON p.policy_id = c.policy_id
    GROUP BY p.plan_type, p.plan_tier
    ORDER BY loss_ratio_pct DESC
""").df()

display(prof_df)


## 2. Top Unprofitable Segments (Loss Ratio > 100%)

In [ ]:
unprof_df = con.execute("""
    WITH seg AS (
        SELECT p.plan_tier, m.age_band, m.region,
               SUM((DATE_DIFF('day', p.policy_start_date, p.policy_end_date) + 1.0)/365.25 * (p.annual_premium/p.members_covered)) AS earned_prem,
               SUM(c.paid_amount) AS incurred
        FROM dim_member m
        JOIN dim_policy p ON m.policy_id = p.policy_id
        LEFT JOIN fact_claims c ON m.member_id = c.member_id
        GROUP BY p.plan_tier, m.age_band, m.region
    )
    SELECT plan_tier, age_band, region,
           ROUND(earned_prem, 2) AS earned_prem,
           ROUND(incurred, 2) AS incurred,
           ROUND(incurred - earned_prem, 2) AS underwriting_deficit,
           ROUND(incurred / earned_prem * 100.0, 2) AS loss_ratio_pct
    FROM seg
    WHERE incurred > earned_prem
    ORDER BY underwriting_deficit DESC
    LIMIT 10
""").df()

display(unprof_df)


## 3. Recommended Underwriting Actions
1. **Bronze Tier Repricing**: Older members (51+) on Bronze plans exhibit >100% loss ratios due to inadequate base premiums. Implement an age-rated premium loading (+25% for age 50+).
2. **Co-pay Enforcement**: Introduce a mandatory 20% co-pay on Tier 1 metro hospital admissions for Bronze and Silver tiers to curb billing inflation.
3. **Preferred Provider Network (PPN)**: Direct volume towards NABH-accredited network facilities with negotiated package rates.


In [ ]:
con.close()